<a href="https://colab.research.google.com/github/mostafa-ja/LLM_from_scratch/blob/main/notebooks/07_guided_tour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guided tour: from text to a generated token

This notebook connects the theory notebooks to the cleaned `llm_from_scratch` package. We will:

1. train a small BPE tokenizer,
2. turn text into next-token training examples,
3. inspect the model's tensor shapes,
4. verify causal masking,
5. run a short training loop,
6. sample text at different temperatures,
7. map the small experiment to GPT-2 124M and FineWeb-Edu.

The model in this tour is intentionally tiny. The point is to see every step, not to produce a great model.


## Step 0: set up Colab

Run the next cell once. It clones the repository and installs the package in editable mode.


In [ ]:
#@title Clone the repository and install the package
import os

REPO = "LLM_from_scratch"
if os.path.basename(os.getcwd()) != REPO:
    if not os.path.isdir(REPO):
        !git clone https://github.com/mostafa-ja/LLM_from_scratch.git
    os.chdir(REPO)

!pip install -q -e .
print("Ready:", os.getcwd())


## Step 1: Byte Pair Encoding (BPE)

BPE starts with 256 byte tokens. It repeatedly finds the most frequent adjacent pair and replaces it with a new token.

The important detail is the **split pattern**. Text is split into chunks first, so a merge never crosses a word boundary. This is why `" hello"` and `"hello"` can tokenize differently.

Our from-scratch tokenizer is in `src/llm_from_scratch/tokenizer.py`.


In [ ]:
from llm_from_scratch.data import download_tiny_shakespeare
from llm_from_scratch.tokenizer import BPETokenizer

path = download_tiny_shakespeare("data/tiny_shakespeare.txt")
text = path.read_text(encoding="utf-8")

# A small corpus keeps BPE training fast. GPT-2 used a much larger corpus
# and learned 50,000 merges; we learn 256 merges here.
tokenizer = BPETokenizer()
tokenizer.train(text[:200_000], vocab_size=512, verbose=False)

print("vocabulary size:", tokenizer.vocab_size)
print("number of merges:", len(tokenizer.merges))
print()
for index, ((left, right), new_id) in enumerate(list(tokenizer.merges.items())[:15], start=1):
    print(index, tokenizer.vocab[left], "+", tokenizer.vocab[right], "->", tokenizer.vocab[new_id], "id", new_id)


In [ ]:
probe = "First Citizen:\nBefore we proceed any further, hear me speak."
ids = tokenizer.encode(probe)

print("characters:", len(probe))
print("tokens:", len(ids))
print("characters per token:", round(len(probe) / len(ids), 2))
print("ids:", ids)
print("pieces:", [tokenizer.vocab[i] for i in ids])
print("round-trip is exact:", tokenizer.decode(ids) == probe)


Notice that the compression ratio is modest. A 512-token vocabulary is tiny compared with GPT-2's 50,257 tokens. More merges mean fewer tokens per sentence, but also a larger embedding matrix.

**Concept check:** why do we start from UTF-8 bytes instead of Unicode characters? Because every possible string can be represented as bytes, so there is no unknown token.


## Step 2: turn text into next-token examples

For training we use `tiktoken`, the production GPT-2 tokenizer. Each training example is a pair:

- `x`: tokens from position `t` to `t + block_size - 1`
- `y`: the same window shifted by one position

The model learns to predict `y[t]` from `x[:t+1]` at every position, in parallel.


In [ ]:
import tiktoken
from llm_from_scratch.data import DataLoaderLite, split_tokens

encoding = tiktoken.get_encoding("gpt2")
tokens = encoding.encode_ordinary(text)
train_tokens, val_tokens = split_tokens(tokens, val_fraction=0.1)

loader = DataLoaderLite(train_tokens, batch_size=4, block_size=16)
x, y = loader.next_batch()

print("x shape:", tuple(x.shape), "y shape:", tuple(y.shape))
print("first context:", repr(encoding.decode(x[0].tolist())))
print("first targets:", repr(encoding.decode(y[0].tolist())))
print("targets are inputs shifted by one:", x[0, 1:].tolist() == y[0, :-1].tolist())


## Step 3: inspect the model

The `nano` preset is deliberately small: 2 layers, 4 heads, 128 embedding dimensions. GPT-2 small uses 12 layers, 12 heads, and 768 dimensions.

The tensor flow is:

| Tensor | Shape | Meaning |
|--------|-------|---------|
| `idx` | `(B, T)` | token ids |
| token embedding | `(B, T, C)` | one vector per token |
| position embedding | `(T, C)` | one vector per position |
| block output | `(B, T, C)` | contextual vector per token |
| logits | `(B, T, vocab_size)` | one score per vocabulary item |
| loss | scalar | mean cross-entropy over all `B * T` positions |


In [ ]:
import torch
from llm_from_scratch.config import GPTConfig
from llm_from_scratch.model import GPT

config = GPTConfig.preset("nano")
model = GPT(config)

print(config)
print(f"parameters: {model.num_parameters():,}")
print("token embedding:", tuple(model.transformer.wte.weight.shape))
print("LM head is tied to token embedding:", model.transformer.wte.weight is model.lm_head.weight)

idx = x[:, :16]
logits, loss = model(idx, idx)
print("input shape:", tuple(idx.shape))
print("logits shape:", tuple(logits.shape))
print("initial loss:", round(loss.item(), 3))
print("random baseline ln(vocab_size):", round(torch.log(torch.tensor(float(config.vocab_size))).item(), 3))


The initial loss is close to `ln(vocab_size)`, which is what you expect from a uniform distribution over the vocabulary. Training should push it below this line.

**Concept check:** why are the logits three-dimensional? Because every position in every sequence produces a prediction. We compare all of them with the shifted targets at once.


### Verify causality

A decoder must not look at future tokens. We can test this directly: change tokens after position 8 and check that logits at positions 0-7 do not change.


In [ ]:
model.eval()
base = x[:1, :16].clone()
changed = base.clone()
changed[:, 8:] = (changed[:, 8:] + 1) % config.vocab_size

with torch.no_grad():
    base_logits, _ = model(base)
    changed_logits, _ = model(changed)

print("past logits unchanged:", torch.allclose(base_logits[:, :8], changed_logits[:, :8]))
print("future logits changed:", not torch.allclose(base_logits[:, 8:], changed_logits[:, 8:]))


This works because `F.scaled_dot_product_attention(..., is_causal=True)` applies a triangular mask. In notebook 02 you can see the same mask built manually with `torch.tril`.


## Step 4: train for a few steps

The training loop in `src/llm_from_scratch/train.py` does the following at every step:

1. set the learning rate from the warmup + cosine schedule,
2. zero the gradients,
3. accumulate gradients over micro-batches,
4. clip the global gradient norm,
5. update the weights with AdamW,
6. periodically evaluate on held-out tokens.

We run only 30 steps. The loss will fall, but the model will still be far from fluent. That is expected.


In [ ]:
from llm_from_scratch.train import TrainConfig, train

config = TrainConfig(
    data_path="data/tiny_shakespeare.txt",
    out_dir="tour_out",
    model="nano",
    block_size=64,
    batch_size=8,
    total_batch_size=512,
    max_steps=30,
    warmup_steps=5,
    max_lr=1e-3,
    eval_interval=5,
    eval_batches=5,
    log_interval=5,
    device="auto",
    sample_prompt="ROMEO:",
    sample_tokens=80,
)

result = train(config)


### Read the numbers

- **Train loss** is measured on the current micro-batches. It is noisy.
- **Validation loss** is measured on tokens the model did not train on. This is the number that matters.
- **Gradient norm** tells you how large the update is before clipping.
- **Tokens per second** is a rough throughput measure.

After only 30 steps the validation loss is still high. A real run uses hundreds of thousands of steps and billions of tokens.


## Step 5: sampling

Sampling turns the next-token distribution into text. Three controls are useful:

- **Temperature** divides the logits before softmax. Lower values make the distribution sharper and more repetitive; higher values make it flatter and more random.
- **Top-k** keeps only the `k` most likely tokens.
- **Top-p (nucleus)** keeps the smallest set of tokens whose cumulative probability reaches `p`.

The model below is undertrained, so the text will be nonsense. The purpose is to see how the controls change the distribution, not to judge quality.


In [ ]:
from llm_from_scratch.generate import load_checkpoint
from llm_from_scratch.train import resolve_device

device = resolve_device("auto")
model, model_config, payload = load_checkpoint("tour_out/best.pt", device)
print("checkpoint step:", payload["step"])
print("checkpoint validation loss:", round(payload["val_loss"], 3))

prompt = "ROMEO:"
prompt_ids = torch.tensor([encoding.encode_ordinary(prompt)], dtype=torch.long, device=device)

for temperature in (0.5, 1.0):
    torch.manual_seed(0)
    output = model.generate(
        prompt_ids,
        max_new_tokens=80,
        temperature=temperature,
        top_k=50,
    )
    print(f"\n--- temperature={temperature} ---")
    print(encoding.decode(output[0].tolist()))


In [ ]:
# Nucleus sampling: keep tokens until their cumulative probability reaches 0.9.
torch.manual_seed(0)
output = model.generate(
    prompt_ids,
    max_new_tokens=80,
    temperature=0.8,
    top_p=0.9,
)
print(encoding.decode(output[0].tolist()))


## Step 6: scale up to GPT-2 124M

The same code scales up by changing the configuration:

| Model | Layers | Heads | Embedding | Parameters |
|-------|--------|-------|-----------|------------|
| nano (this tour) | 2 | 4 | 128 | ~6.9M |
| GPT-2 small | 12 | 12 | 768 | ~124M |
| GPT-2 medium | 24 | 16 | 1024 | ~350M |
| GPT-2 large | 36 | 20 | 1280 | ~760M |
| GPT-2 XL | 48 | 25 | 1600 | ~1.5B |

The full GPT-2 recipe adds:

- vocabulary padding from 50,257 to 50,304 (a multiple of 64),
- scaled residual initialization,
- weight tying,
- AdamW with `betas=(0.9, 0.95)`,
- gradient clipping at 1.0,
- linear warmup and cosine decay,
- gradient accumulation to reach a 524,288-token batch,
- bf16 autocast, `torch.compile`, fused AdamW, and FlashAttention.

Notebook 03 explains each optimization. Notebook 04 runs the training loop. Notebook 06 prepares FineWeb-Edu shards for larger runs.

```bash
# local quick run with the reference package
python scripts/train.py --model tiny --max-steps 600

# generate from the best checkpoint
python scripts/generate.py --checkpoint out/best.pt --prompt "ROMEO:"
```


## Exercises

1. Train the BPE tokenizer with `vocab_size=1024` and compare the number of tokens for the same probe sentence.
2. Change `block_size` from 64 to 128. Which tensors change shape? Which do not?
3. Remove the weight-tying line in `model.py`. How many parameters are added?
4. Replace the `F.scaled_dot_product_attention` call with manual `Q @ K.T`, masking, softmax, and `@ V`. Compare outputs in eval mode.
5. Train for 300 steps instead of 30. Plot train and validation loss. Does validation loss start to rise?
6. Add dropout with `dropout=0.1` and compare validation loss.
7. Try top-p sampling with `top_p=0.5` and `top_p=1.0`. What changes?
8. Read `tests/test_model.py` and add a test for your favorite property.


## Troubleshooting

**CUDA out of memory.** Reduce `batch_size` or `block_size`. Increase `total_batch_size` only if memory allows; use gradient accumulation to keep the effective batch large.

**CPU is slow.** In Colab, choose *Runtime -> Change runtime type -> GPU*. The CPU path is correct, just slower.

**Package import fails.** Re-run the setup cell. Make sure the working directory is the repository root.

**Colab has an old copy.** Run `!git pull` in the repository, then re-run setup.

**Training loss is not falling.** Check the learning rate, warmup, and tokenization. A loss near `ln(vocab_size)` means the model is still near random.
